In [1]:
!pip install -q gradio scikit-learn numpy pandas matplotlib

In [2]:
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split


def visualize_linear_regression(n_points, noise, test_size):

    # 1. Generate a dataset
    np.random.seed(42)

    X = np.linspace(0, 10, int(n_points)).reshape(-1, 1)
    y = 3 * X.ravel() + 5 + np.random.normal(
        0, noise, int(n_points)
    )

    # 2. Split into training and testing data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=42
    )

    # 3. Train the actual ML model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # 4. Make predictions
    predictions = model.predict(X_test)

    # 5. Calculate evaluation metrics
    r2 = r2_score(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)

    # 6. Plot data and regression line
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.scatter(
        X_train.ravel(), y_train,
        label="Training data"
    )

    ax.scatter(
        X_test.ravel(), y_test,
        marker="x",
        s=80,
        label="Testing data"
    )

    line_x = np.linspace(0, 10, 200).reshape(-1, 1)
    line_y = model.predict(line_x)

    ax.plot(
        line_x.ravel(), line_y,
        linewidth=2,
        label="Regression line"
    )

    ax.set_title("Linear Regression Visualizer")
    ax.set_xlabel("Input Feature (X)")
    ax.set_ylabel("Target Value (y)")
    ax.legend()
    ax.grid(alpha=0.25)

    metrics = f"""
### Model Results

- **Slope:** {model.coef_[0]:.3f}
- **Intercept:** {model.intercept_:.3f}
- **R² Score:** {r2:.3f}
- **Mean Squared Error:** {mse:.3f}
- **Training samples:** {len(X_train)}
- **Testing samples:** {len(X_test)}
"""

    explanation = """
### How Linear Regression Works

1. Generate sample data with an approximately linear relationship.
2. Divide the data into training and testing sets.
3. Train a Linear Regression model on the training data.
4. Find the best-fit line by minimizing the sum of squared residuals.
5. Evaluate predictions on the held-out testing data.

**Try changing the noise level:** More noise generally makes the relationship harder to predict.
"""

    return fig, metrics, explanation


with gr.Blocks(theme=gr.themes.Soft(), title="ML Algorithm Visualizer") as demo:

    gr.Markdown(
        """
        # 🧠 ML Algorithm Visualizer
        ### Learn Machine Learning Through Interactive Experiments
        """
    )

    gr.Markdown("## Linear Regression")

    with gr.Row():

        with gr.Column(scale=1):

            n_points = gr.Slider(
                minimum=30,
                maximum=300,
                value=100,
                step=10,
                label="Number of Data Points"
            )

            noise = gr.Slider(
                minimum=0,
                maximum=15,
                value=3,
                step=0.5,
                label="Noise Level"
            )

            test_size = gr.Slider(
                minimum=0.1,
                maximum=0.4,
                value=0.2,
                step=0.05,
                label="Test Data Proportion"
            )

            run_button = gr.Button(
                "Train Model & Visualize",
                variant="primary"
            )

        with gr.Column(scale=2):

            plot_output = gr.Plot(label="Regression Visualization")

            metrics_output = gr.Markdown()

    explanation_output = gr.Markdown()

    run_button.click(
        fn=visualize_linear_regression,
        inputs=[n_points, noise, test_size],
        outputs=[plot_output, metrics_output, explanation_output]
    )

    demo.load(
        fn=visualize_linear_regression,
        inputs=[n_points, noise, test_size],
        outputs=[plot_output, metrics_output, explanation_output]
    )


demo.launch(share=True)

/tmp/ipykernel_618/2070577747.py:94: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="ML Algorithm Visualizer") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b35127e0fcac7dbba5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
import gradio as gr
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.inspection import DecisionBoundaryDisplay


def classification_visualizer(algorithm, complexity, n_points):

    X, y = make_classification(
        n_samples=int(n_points),
        n_features=2,
        n_informative=2,
        n_redundant=0,
        n_clusters_per_class=1,
        class_sep=1.3,
        random_state=42
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    if algorithm == "Logistic Regression":
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=complexity, random_state=42)
        )

    elif algorithm == "SVM":
        model = make_pipeline(
            StandardScaler(),
            SVC(C=complexity, kernel="rbf")
        )

    elif algorithm == "Decision Tree":
        model = DecisionTreeClassifier(
            max_depth=int(complexity),
            random_state=42
        )

    else:
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=int(complexity),
            random_state=42
        )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    fig, ax = plt.subplots(figsize=(8, 5))

    DecisionBoundaryDisplay.from_estimator(
        model,
        X,
        response_method="predict",
        plot_method="contourf",
        alpha=0.25,
        ax=ax
    )

    ax.scatter(
        X_train[:, 0], X_train[:, 1],
        c=y_train, edgecolors="black",
        label="Training data"
    )

    ax.scatter(
        X_test[:, 0], X_test[:, 1],
        c=y_test, marker="x", s=80,
        label="Testing data"
    )

    ax.set_title(f"{algorithm} Decision Boundary")
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    ax.legend()

    metrics = f"""
## {algorithm} Results

- **Test Accuracy:** {accuracy:.3f}
- **Training samples:** {len(X_train)}
- **Testing samples:** {len(X_test)}

### How it works

"""

    explanations = {
        "Logistic Regression":
            "Estimates class probabilities using a logistic function and separates classes with a learned decision boundary.",

        "SVM":
            "Finds a decision boundary that separates classes while maximizing the margin. The RBF kernel can model nonlinear boundaries.",

        "Decision Tree":
            "Splits the data into smaller groups using feature-based rules. The maximum depth controls tree complexity.",

        "Random Forest":
            "Combines many decision trees trained with randomness and aggregates their predictions."
    }

    metrics += explanations[algorithm]

    if algorithm == "Decision Tree":
        tree_model = model
    elif algorithm == "Random Forest":
        tree_model = model
    else:
        tree_model = None

    return fig, metrics


def show_tree(n_points, depth):

    X, y = make_classification(
        n_samples=int(n_points),
        n_features=2,
        n_informative=2,
        n_redundant=0,
        random_state=42
    )

    model = DecisionTreeClassifier(
        max_depth=int(depth),
        random_state=42
    )
    model.fit(X, y)

    fig, ax = plt.subplots(figsize=(14, 7))

    plot_tree(
        model,
        feature_names=["Feature 1", "Feature 2"],
        class_names=["Class 0", "Class 1"],
        filled=True,
        rounded=True,
        ax=ax
    )

    ax.set_title("Decision Tree Structure")

    return fig


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="ML Algorithm Visualizer"
) as demo:

    gr.Markdown(
        """
# 🧠 ML Algorithm Visualizer
### Explore seven machine learning algorithms interactively
"""
    )

    gr.Markdown(
        """
Your existing notebook has Linear Regression, KNN and K-Means.
This module adds Logistic Regression, SVM, Decision Tree and Random Forest.
"""
    )

    with gr.Tab("Classification Algorithms"):

        algorithm = gr.Dropdown(
            choices=[
                "Logistic Regression",
                "SVM",
                "Decision Tree",
                "Random Forest"
            ],
            value="Logistic Regression",
            label="Choose an Algorithm"
        )

        complexity = gr.Slider(
            minimum=1,
            maximum=10,
            value=5,
            step=1,
            label="C (Logistic Regression/SVM) or Max Depth (Trees)"
        )

        n_points = gr.Slider(
            minimum=50,
            maximum=300,
            value=150,
            step=25,
            label="Number of Data Points"
        )

        run_button = gr.Button("Run Algorithm", variant="primary")
        plot_output = gr.Plot()
        result_output = gr.Markdown()

        run_button.click(
            classification_visualizer,
            inputs=[algorithm, complexity, n_points],
            outputs=[plot_output, result_output]
        )

    with gr.Tab("Decision Tree Structure"):

        tree_points = gr.Slider(
            minimum=50, maximum=300,
            value=150, step=25,
            label="Number of Data Points"
        )

        tree_depth = gr.Slider(
            minimum=1, maximum=6,
            value=3, step=1,
            label="Maximum Tree Depth"
        )

        tree_button = gr.Button("Visualize Decision Tree")
        tree_output = gr.Plot()

        tree_button.click(
            show_tree,
            inputs=[tree_points, tree_depth],
            outputs=tree_output
        )

demo.launch(share=True)

/tmp/ipykernel_618/1618176217.py:164: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://620246a19f9107e428.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
